# 3 - Out-of-sample backtest (Jan-Jun 2026)

Train a **price-only** direction classifier on the 2025 data (`btc_features.csv`) and test it on a completely separate period, **Jan-Jun 2026** (`btc_backtest_features.csv`, built by `make_backtest_features.py`).

Why price-only: the news/sentiment dataset only covers 2025-07..2025-11, so 2026 has no news features. This is the honest generalization test -- a different period, no regime overlap.

Run `make_backtest_features.py` first to create the 2026 file.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

train = pd.read_csv('btc_features.csv', index_col=0, parse_dates=True).sort_index()
btest = pd.read_csv('btc_backtest_features.csv', index_col=0, parse_dates=True).sort_index()
print('train (2025):', train.shape, train.index.min().date(), '->', train.index.max().date())
print('backtest (2026):', btest.shape, btest.index.min().date(), '->', btest.index.max().date())

## Setup (price-only features)

In [ ]:
price_features = ['close', 'volume', 'ret_1h', 'ret_3h', 'vol_change',
                  'rsi_14', 'macd', 'macd_signal', 'sma20_ratio', 'bb_pct']
HORIZONS = [6, 12]

clf_factories = {
    'logreg': lambda: make_pipeline(StandardScaler(),
                                    LogisticRegression(max_iter=1000, class_weight='balanced')),
    'rf': lambda: RandomForestClassifier(n_estimators=300, random_state=0, class_weight='balanced'),
    'xgb': lambda: XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                 eval_metric='logloss', random_state=0),
}

def make_xy(d, h):
    future = d['close'].shift(-h)
    y = (future > d['close']).astype(int).values
    mask = future.notna().values
    return d[price_features].values[mask], y[mask], d.index[mask]

## Train on 2025 -> predict 2026

In [ ]:
rows = []
for h in HORIZONS:
    Xtr, ytr, _ = make_xy(train, h)
    Xbt, ybt, _ = make_xy(btest, h)
    base = max(ybt.mean(), 1 - ybt.mean())
    for mname, mk in clf_factories.items():
        m = mk().fit(Xtr, ytr)
        p = m.predict(Xbt)
        proba = m.predict_proba(Xbt)[:, 1]
        rows.append({'horizon': f'{h}h', 'model': mname,
                     'acc': round(accuracy_score(ybt, p), 3),
                     'auc': round(roc_auc_score(ybt, proba), 3) if len(set(ybt)) > 1 else np.nan,
                     'baseline_2026': round(base, 3)})
backtest_results = pd.DataFrame(rows)
backtest_results

## Visualize predictions on 2026

In [ ]:
VIZ_MODEL = 'xgb'
VIZ_HORIZON = 6

Xtr, ytr, _ = make_xy(train, VIZ_HORIZON)
Xbt, ybt, idx_bt = make_xy(btest, VIZ_HORIZON)
m = clf_factories[VIZ_MODEL]().fit(Xtr, ytr)
proba = m.predict_proba(Xbt)[:, 1]
pred = (proba > 0.5).astype(int)
close_bt = btest['close'].values[btest['close'].shift(-VIZ_HORIZON).notna().values]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(idx_bt, close_bt, color='black', lw=1.0, zorder=1)
up = pred == 1
ax.scatter(idx_bt[up], close_bt[up], c='green', s=10, label='pred up', zorder=2)
ax.scatter(idx_bt[~up], close_bt[~up], c='red', s=10, label='pred down', zorder=2)
ax.set_title(f'{VIZ_MODEL.upper()} {VIZ_HORIZON}h predicted direction on 2026 (out-of-sample)')
ax.set_ylabel('BTC close'); ax.legend(); plt.tight_layout(); plt.show()

ConfusionMatrixDisplay(confusion_matrix(ybt, pred), display_labels=['down', 'up']).plot(cmap='Blues')
plt.title(f'{VIZ_MODEL.upper()} {VIZ_HORIZON}h 2026 confusion matrix'); plt.tight_layout(); plt.show()
print('2026 accuracy:', round((pred == ybt).mean(), 3),
      '| AUC:', round(roc_auc_score(ybt, proba), 3),
      '| 2026 up-rate:', round(ybt.mean(), 3))

## Simple trading backtest (illustrative)
Turn the directional call into a position rebalanced hourly (long if pred up, short if down), held 1h. **Ignores trading fees** -- hourly rebalancing in reality would incur heavy costs, so read this as a signal-quality check, not a realistic P&L.

In [ ]:
h = VIZ_HORIZON
bt = btest.copy()
future = bt['close'].shift(-h)
mask = future.notna().values
sig = pd.Series(np.where(pred == 1, 1.0, -1.0), index=idx_bt)        # +1 long / -1 short
next_ret = bt['ret_1h'].shift(-1).reindex(idx_bt)                    # next-hour return
strat = (sig * next_ret).dropna()
bh = next_ret.reindex(strat.index)
eq_strat = (1 + strat).cumprod()
eq_bh = (1 + bh).cumprod()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(eq_strat.index, eq_strat.values, label=f'strategy ({VIZ_MODEL} {h}h)', color='tab:blue')
ax.plot(eq_bh.index, eq_bh.values, label='buy & hold', color='black')
ax.axhline(1.0, ls=':', c='grey')
ax.set_title('Equity curve on 2026 (no fees) - illustrative')
ax.set_ylabel('growth of $1'); ax.legend(); plt.tight_layout(); plt.show()
print('strategy total return:', round(eq_strat.iloc[-1] - 1, 3),
      '| buy&hold:', round(eq_bh.iloc[-1] - 1, 3))

## Interpretation

- Compare 2026 **accuracy/AUC vs the 2026 baseline**. If it's ~baseline / AUC ~0.5, the price model **does not generalize** out-of-sample -- the expected result, confirming no robust edge.
- The equity curve (fees ignored) is a sanity check only; beating buy & hold here without fees is weak evidence, and hourly rebalancing would be eaten alive by costs in practice.
- This backtest can't test the news/sentiment model (no 2026 news). To do that you'd need 2026 news data.